In [ ]:
!pip install -q faiss-cpu chromadb sentence-transformers beautifulsoup4 lxml

import requests
import os

os.makedirs('filings', exist_ok=True)

# SEC requires a descriptive User-Agent header identifying who's accessing their servers
headers = {'User-Agent': 'Research Project noorulsehar2004@gmail.com'}

filings = {
    'apple_10k.htm': 'https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm',
    'microsoft_10k.htm': 'https://www.sec.gov/Archives/edgar/data/789019/000095017025100235/msft-20250630.htm',
    'tesla_10k.htm': 'https://www.sec.gov/Archives/edgar/data/1318605/000162828025003063/tsla-20241231.htm',
}

for filename, url in filings.items():
    r = requests.get(url, headers=headers)
    with open(f'filings/{filename}', 'wb') as f:
        f.write(r.content)
    print(filename, '-', len(r.content), 'bytes, status', r.status_code)

apple_10k.htm - 1520208 bytes, status 200
microsoft_10k.htm - 8158067 bytes, status 200
tesla_10k.htm - 2596459 bytes, status 200


In [ ]:
#raw html structure
with open('filings/apple_10k.htm', 'r', encoding='utf-8') as f:
    raw_html = f.read()

print("Total characters:", len(raw_html))
print()
print("First 2000 characters:")
print(raw_html[:2000])

Total characters: 1520208

First 2000 characters:
<?xml version='1.0' encoding='ASCII'?>
<!--XBRL Document Created with the Workiva Platform-->
<!--Copyright 2025 Workiva-->
<!--r:b93d322a-f6d3-4356-8a13-e3e1c42e12bd,g:44705cc3-5a3a-440a-975b-ed1d3694d858,d:719388195b384d85a4e238ad88eba90a-->
<html xmlns="http://www.w3.org/1999/xhtml" xmlns:dei="http://xbrl.sec.gov/dei/2025" xmlns:link="http://www.xbrl.org/2003/linkbase" xmlns:country="http://xbrl.sec.gov/country/2025" xmlns:ixt="http://www.xbrl.org/inlineXBRL/transformation/2020-02-12" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:aapl="http://www.apple.com/20250927" xmlns:xbrli="http://www.xbrl.org/2003/instance" xmlns:xbrldi="http://xbrl.org/2006/xbrldi" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:ix="http://www.xbrl.org/2013/inlineXBRL" xmlns:cyd="http://xbrl.sec.gov/cyd/2025" xmlns:srt="http://fasb.org/srt/2025" xmlns:iso4217="http://www.xbrl.org/2003/iso4217" xmlns:ixt-sec="http://www.sec.gov/inlineXBRL/transfo

In [ ]:
from bs4 import XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

In [ ]:
#clean extraction
from bs4 import BeautifulSoup
def extract_clean_text(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        raw_html = f.read()

    soup = BeautifulSoup(raw_html, 'lxml')
    #removes hidden XBRL metadata blocks, not real content and will display none
    for hidden in soup.find_all(style=lambda s: s and 'display:none' in s.replace(' ', '')):
        hidden.decompose()
    #removing  script/style tags entirely
    for tag in soup(['script', 'style']):
        tag.decompose()

    text = soup.get_text(separator='\n')
    #collapse excessive blank lines
    lines = [line.strip() for line in text.split('\n')]
    lines = [line for line in lines if line]
    clean_text = '\n'.join(lines)
    return clean_text

apple_text = extract_clean_text('filings/apple_10k.htm')
print("Clean text length:", len(apple_text))
print()
print("First 1500 characters:")
print(apple_text[:1500])

Clean text length: 206827

First 1500 characters:
aapl-20250927
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM
10-K
(Mark One)
☒
ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended
September 27
, 2025
or
☐
TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from
to
.
Commission File Number:
001-36743
Apple Inc.
(Exact name of Registrant as specified in its charter)
California
94-2404110
(State or other jurisdiction
of incorporation or organization)
(I.R.S. Employer Identification No.)
One Apple Park Way
Cupertino
,
California
95014
(Address of principal executive offices)
(Zip Code)
(
408
)
996-1010
(Registrant’s telephone number, including area code)
Securities registered pursuant to Section 12(b) of the Act:
Title of each class
Trading symbol(s)
Name of each exchange on which registered
Common Stock, $0.00001 par value per share
AAPL


In [ ]:
#extraxting all three files and focusing total length
microsoft_text = extract_clean_text('filings/microsoft_10k.htm')
tesla_text = extract_clean_text('filings/tesla_10k.htm')
documents = {
    'Apple 10-K (FY2025)':apple_text,
    'Microsoft 10-K (FY2025)':microsoft_text,
    'Tesla 10-K (FY2024)':tesla_text,
}

for name, text in documents.items():
    print(f"{name}: {len(text):,} characters (~{len(text)//5:,} words)")

Apple 10-K (FY2025): 206,827 characters (~41,365 words)
Microsoft 10-K (FY2025): 313,444 characters (~62,688 words)
Tesla 10-K (FY2024): 385,450 characters (~77,090 words)


In [ ]:
#chuncking strategy 1: fixed size character chunks with overlap
def fixed_size_chunks(text, chunk_size=1000, overlap=200):
#Naive chunking just slice text into fixed character windows,with some overlap so we don't cut sentences awkwardly at boundaries."""
    chunks = []
    start = 0
    while start<len(text):
        end=start+chunk_size
        chunk=text[start:end]
        chunks.append(chunk)
        start+=chunk_size-overlap  #move forward but overlap with previous chunk
    return chunks

#testing on Apple's filing first
apple_chunks_fixed = fixed_size_chunks(apple_text, chunk_size=1000, overlap=200)
print(f"Apple 10-K split into {len(apple_chunks_fixed)} fixed-size chunks")
print()
print("Example chunk (#50):")
print(repr(apple_chunks_fixed[50]))

Apple 10-K split into 259 fixed-size chunks

Example chunk (#50):
'cruit and retain highly skilled personnel to execute on its strategic initiatives, and the timely and successful development and market acceptance of new products, services and technologies. Success also relies on the Company’s ability to manage the risks associated with new technologies and production ramp-up issues, the effective integration of third-party services and technologies into the Company’s products and services, the availability, delivery and performance of application software or other third-party support for the Company’s products and services, the effective management of manufacturing and other purchase commitments and the management of inventory levels in line with anticipated product demand, and the availability of products in appropriate quantities and at expected costs to meet anticipated demand. Additionally, quality issues or other defects or deficiencies can adversely affect the success of new pro

In [ ]:
#built embeddings and faiss index for the naive chunking approcah
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
embed_model=SentenceTransformer('all-MiniLM-L6-v2')
#embed all chunks
apple_embeddings=embed_model.encode(apple_chunks_fixed, show_progress_bar=True)
print("Embedding shape:", apple_embeddings.shape)
#build faiss index (L2 distance for now)
dimension=apple_embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
index.add(np.array(apple_embeddings).astype('float32'))
print("FAISS index built with", index.ntotal, "vectors")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embedding shape: (259, 384)
FAISS index built with 259 vectors


In [ ]:
def search(query, index, chunks, model, k=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(np.array(query_embedding).astype('float32'), k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'distance': distances[0][rank],
            'chunk': chunks[idx]
        })
    return results

In [ ]:
#faiss doesn't have a dedicated "cosine similarity" index type directly the standard trick is normalizes every vector to unit length, then use
# Inner Product (dot product). For normalized vectors dot product==cosine similarity

def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / norms
#built second FAISS index using cosine similarity via normalized dot product
apple_embeddings_norm=normalize(np.array(apple_embeddings)).astype('float32')
index_cosine = faiss.IndexFlatIP(dimension) #IP means Inner Product
index_cosine.add(apple_embeddings_norm)

def search_cosine(query, index, chunks, model, k=3):
    query_embedding=model.encode([query])
    query_embedding_norm=normalize(np.array(query_embedding)).astype('float32')
    scores,indices = index.search(query_embedding_norm, k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'score': scores[0][rank],  #higher=more similar for cosine
            'chunk': chunks[idx]
        })
    return results

query="What was Apple's total net sales revenue?"
print("=" * 60)
print("L2 DISTANCE RESULTS")
print("=" * 60)
results_l2 = search(query, index, apple_chunks_fixed, embed_model, k=3)
for r in results_l2:
    print(f"Rank {r['rank']} (L2 distance: {r['distance']:.4f})")
    print(r['chunk'][:250])
    print()

print("=" * 60)
print("COSINE SIMILARITY RESULTS")
print("=" * 60)
results_cosine = search_cosine(query, index_cosine, apple_chunks_fixed, embed_model, k=3)
for r in results_cosine:
    print(f"Rank {r['rank']} (cosine score: {r['score']:.4f})")
    print(r['chunk'][:250])
    print()

#did the two metrics pick the same chunks, just in different order or not at all?
l2_chunk_ids = set(r['chunk'][:50] for r in results_l2)
cosine_chunk_ids = set(r['chunk'][:50] for r in results_cosine)
print("Same top-3 chunks (by content overlap)?", l2_chunk_ids == cosine_chunk_ids)

L2 DISTANCE RESULTS
Rank 1 (L2 distance: 0.6001)
586
$
201,183
$
200,583
Mac
33,708
29,984
29,357
iPad
28,023
26,694
28,300
Wearables, Home and Accessories
35,686
37,005
39,845
Services
(1)
109,158
96,169
85,200
Total net sales
$
416,161
$
391,035
$
383,285
Portion of total net sales that was inclu

Rank 2 (L2 distance: 0.7855)
ber 28, 2024, the Company had
two
vendors that individually represented 10% or more of total vendor non-trade receivables, which accounted for
44
% and
23
%.
Note 5 –
Property, Plant and Equipment
The following table shows the Company’s gross propert

Rank 3 (L2 distance: 0.7965)
multiple factors when determining whether it obtains control of third-party products, including evaluating if it can establish the price of the product, retains inventory risk for tangible products or has the responsibility for ensuring acceptability

COSINE SIMILARITY RESULTS
Rank 1 (cosine score: 0.6999)
586
$
201,183
$
200,583
Mac
33,708
29,984
29,357
iPad
28,023
26,694
28,300
Wearab

## Distance metric comparison: L2 vs Cosine Similarity

Tested the same query ("What was Apple's total net sales revenue?") against both an L2 index
and a cosine similarity index (via normalized dot product).

**Result:** identical top-3 chunks, in the same order, for both metrics.

**Why they agreed here:** `all-MiniLM-L6-v2` tends to produce embeddings with fairly similar
magnitudes to begin with, so L2 and cosine ranked chunks almost the same way in this case. This
isn't guaranteed in general L2 and cosine can diverge when embedding magnitudes vary more —
but for this model and this dataset, the choice of metric didn't change the outcome.

**More important finding naive chunking breaks tables:** the top-ranked chunk *did* contain
the correct answer (Total net sales: $416,161 million), but the chunk itself is a broken
financial table with no column headers:

> "586 $ 201,183 $ 200,583 Mac 33,708 29,984 29,357 iPad 28,023 26,694 28,300..."

The fixed-size chunker sliced straight through the middle of a table, losing the header row
(which years/columns these numbers belong to). The correct number is present, but a reader (or
an LLM generating an answer from this chunk alone) would have no reliable way to know what
these numbers actually represent without the missing header context. This is a direct,
concrete example of the exact problem the task description flagged — naive chunking breaks on
tables.

In [ ]:
#Chunking strategy #2 recursive, boundary aware splitting
import re
def recursive_chunks(text, chunk_size=1000, overlap=200):
    """Boundary-aware chunking: tries to split on paragraph breaks first,
    then sentence breaks, only falling back to a hard cut if neither exists
    within a reasonable window. This avoids cutting mid-sentence/mid-table-row."""

    # Split into paragraphs first (blank line separated blocks)
    paragraphs = [p for p in text.split('\n') if p.strip()]

    chunks = []
    current_chunk = ""

    for para in paragraphs:
        #if adding this paragraph would exceed chunk_size, finalize current chunk
        if len(current_chunk) + len(para) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            #start new chunk with overlap: carry over the last bit of the previous chunk
            overlap_text = current_chunk[-overlap:] if len(current_chunk) > overlap else current_chunk
            current_chunk = overlap_text + "\n" + para
        else:
            current_chunk += "\n" + para if current_chunk else para

        # If a single paragraph itself is longer than chunk_size, split it by sentences
        if len(current_chunk) > chunk_size * 1.5:
            sentences = re.split(r'(?<=[.!?])\s+', current_chunk)
            temp = ""
            for sent in sentences:
                if len(temp) + len(sent) > chunk_size and temp:
                    chunks.append(temp.strip())
                    temp = sent
                else:
                    temp += " " + sent if temp else sent
            current_chunk = temp

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

apple_chunks_recursive = recursive_chunks(apple_text, chunk_size=1000, overlap=200)
print(f"Apple 10-K split into {len(apple_chunks_recursive)} recursive/boundary-aware chunks")
print(f"(compare: {len(apple_chunks_fixed)} chunks with naive fixed-size chunking)")
print()
print("Example chunk (revenue table, if we can find it):")
for c in apple_chunks_recursive:
    if 'Total net sales' in c:
        print(c[:600])
        break

Apple 10-K split into 300 recursive/boundary-aware chunks
(compare: 259 chunks with naive fixed-size chunking)

Example chunk (revenue table, if we can find it):
Trade and other international disputes can have an adverse impact on the overall macroeconomic environment and result in shifts and reductions in consumer spending and negative consumer sentiment for the Company’s products and services, all of which can further adversely affect the Company’s business and results of operations.
Segment Operating Performance
The following table shows net sales by reportable segment for 2025, 2024 and 2023 (dollars in millions):
2025
Change
2024
Change
2023
Americas
$
178,353
7
%
$
167,045
3
%
$
162,560
Europe
111,032
10
%
101,328
7
%
94,294
Greater China
64,377



In [ ]:
print("FIXED SIZE CHUNKING: chunk containing '416,161'")
for c in apple_chunks_fixed:
    if '416,161' in c:
        print(c[:600])
        print()
        break

print("RECURSIVE CHUNKING: chunk containing '416,161'")
for c in apple_chunks_recursive:
    if '416,161' in c:
        print(c[:600])
        print()
        break

FIXED SIZE CHUNKING: chunk containing '416,161'
ng Performance
The following table shows net sales by reportable segment for 2025, 2024 and 2023 (dollars in millions):
2025
Change
2024
Change
2023
Americas
$
178,353
7
%
$
167,045
3
%
$
162,560
Europe
111,032
10
%
101,328
7
%
94,294
Greater China
64,377
(4)
%
66,952
(8)
%
72,559
Japan
28,703
15
%
25,052
3
%
24,257
Rest of Asia Pacific
33,696
10
%
30,658
4
%
29,615
Total net sales
$
416,161
6
%
$
391,035
2
%
$
383,285
Americas
Americas net sales increased during 2025 compared to 2024 primarily due to higher net sales of iPhone and Services. The weakness in foreign currencies relative to the U

RECURSIVE CHUNKING: chunk containing '416,161'
Trade and other international disputes can have an adverse impact on the overall macroeconomic environment and result in shifts and reductions in consumer spending and negative consumer sentiment for the Company’s products and services, all of which can further adversely affect the Company’s business a

## Chunking Strategy #2: Recursive/boundary aware splitting

Instead of blind fixed size character slicing, this strategy splits on paragraph boundaries
first, falling back to sentence boundaries only when a paragraph is too long. Result: 300
chunks (vs 259 for fixed-size): slightly more chunks, since respecting boundaries means less
tight packing.

**Direct comparison on the same table** (Apple's segment revenue breakdown, containing the
figure $416,161M total net sales):
 **Fixed-size:** chunk starts mid word ("ng Performance...", cut from "Segment Operating
  Performance"). The table header survived only by chance the character boundary happened to
  land nearby.
  **Recursive:** chunk starts at a clean sentence boundary and flows naturally into the table
  with its header fully intact by design, not luck.

**Honest limitation:** neither strategy fully solves table structure both still flatten the
actual table into a single run of numbers rather than preserving rows/columns. Recursive
chunking's real advantage is avoiding mid-sentence/mid-word cuts, not fixing tables outright.
A proper fix would need table-aware parsing (e.g., extracting `<table>` elements separately
before chunking the surrounding prose) a natural next improvement beyond this week's scope.

In [ ]:
#Chunk all 3 documents and track which company each chunk came from
#Chunk Microsoft and Tesla the same way
microsoft_chunks = recursive_chunks(microsoft_text, chunk_size=1000, overlap=200)
tesla_chunks = recursive_chunks(tesla_text, chunk_size=1000, overlap=200)
#build one combined list, tracking source metadata for each chunk
all_chunks = []
all_metadata = []

for chunk in apple_chunks_recursive:
    all_chunks.append(chunk)
    all_metadata.append({'company': 'Apple', 'filing': 'Apple 10-K (FY2025)'})

for chunk in microsoft_chunks:
    all_chunks.append(chunk)
    all_metadata.append({'company': 'Microsoft', 'filing': 'Microsoft 10-K (FY2025)'})

for chunk in tesla_chunks:
    all_chunks.append(chunk)
    all_metadata.append({'company': 'Tesla', 'filing': 'Tesla 10-K (FY2024)'})

print(f"Total chunks across all 3 filings: {len(all_chunks)}")
print(f"  Apple: {len(apple_chunks_recursive)}")
print(f"  Microsoft: {len(microsoft_chunks)}")
print(f"  Tesla: {len(tesla_chunks)}")

Total chunks across all 3 filings: 1345
  Apple: 300
  Microsoft: 471
  Tesla: 574


In [ ]:
#combined faiss index with metadata
# Embed all chunks from all 3 companies at once
all_embeddings=embed_model.encode(all_chunks, show_progress_bar=True)
print("Embedding shape:", all_embeddings.shape)
#Normalize for cosine similarity
all_embeddings_norm=normalize(np.array(all_embeddings)).astype('float32')
combined_index=faiss.IndexFlatIP(dimension)
combined_index.add(all_embeddings_norm)
print("Combined FAISS index built with", combined_index.ntotal, "vectors across 3 companies")

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Embedding shape: (1345, 384)
Combined FAISS index built with 1345 vectors across 3 companies


In [ ]:
def retrieve_with_citations(query, index, chunks, metadata, model, k=3, similarity_threshold=0.35):
#Retrieve top k chunks for a query, attaching citation info

    query_embedding = model.encode([query])
    query_embedding_norm = normalize(np.array(query_embedding)).astype('float32')
    scores, indices = index.search(query_embedding_norm, k)
    results = []
    for rank, idx in enumerate(indices[0]):
        score = scores[0][rank]
        results.append({
            'rank': rank + 1,
            'score': float(score),
            'chunk': chunks[idx],
            'company': metadata[idx]['company'],
            'filing': metadata[idx]['filing'],
        })

    #If even the best match is weak, flag it — this is the "I don't know" trigger
    best_score=results[0]['score'] if results else 0
    confident=best_score >= similarity_threshold
    return results, confident

def answer_question(query, index, chunks, metadata, model, k=3):
    results, confident = retrieve_with_citations(query, index, chunks, metadata, model, k=k)
    print(f"QUERY: {query}\n")
    if not confident:
        print("ANSWER: Not found in the provided documents.")
        print(f"  (Best match similarity: {results[0]['score']:.3f}, below threshold)")
        return

    print("Retrieved supporting evidence:\n")
    for r in results:
        print(f"[{r['rank']}] {r['filing']} (similarity: {r['score']:.3f})")
        print(f"    \"{r['chunk'][:300]}...\"")
        print()

# Test with a real, answerable question
answer_question("What was Apple's total net sales revenue?", combined_index, all_chunks, all_metadata, embed_model)

QUERY: What was Apple's total net sales revenue?

Retrieved supporting evidence:

[1] Apple 10-K (FY2025) (similarity: 0.651)
    "9
13
%
85,200
Total net sales
$
416,161
6
%
$
391,035
2
%
$
383,285
(1)
Services net sales include amortization of the deferred value of services bundled in the sales price of certain products.
iPhone
iPhone net sales increased during 2025 compared to 2024 due to higher net sales of Pro models.
Mac
..."

[2] Apple 10-K (FY2025) (similarity: 0.642)
    "efore transferring it to the customer. Therefore, the Company accounts for all third-party application–related sales on a net basis by recognizing in Services net sales only the commission it retains.
Apple Inc. | 2025 Form 10-K | 35
The following table shows disaggregated net sales, as well as the ..."

[3] Apple 10-K (FY2025) (similarity: 0.625)
    "deferred revenue as of the beginning of the period
$
8,229
$
7,728
$
8,169
(1)
Services net sales include amortization of the deferred value of services bundle

In [ ]:
#question these documents genuinely cannot answer
answer_question("What is the capital of France?", combined_index, all_chunks, all_metadata, embed_model)
print("\n" + "="*60 + "\n")
#question about a company NOT in our document set
answer_question("What was Amazon's total net sales revenue?", combined_index, all_chunks, all_metadata, embed_model)

QUERY: What is the capital of France?

ANSWER: Not found in the provided documents.
  (Best match similarity: 0.215, below threshold)


QUERY: What was Amazon's total net sales revenue?

Retrieved supporting evidence:

[1] Microsoft 10-K (FY2025) (similarity: 0.567)
    "revenue, operating expenses, and operating income were as follows during the periods presented:
(In millions)
Year Ended June 30,
2025
2024
2023
Productivity and Business Processes
Revenue
$
120,810
$
106,820
$
94,151
Cost of revenue
22,422
19,611
17,202
Operating expenses
28,615
27,548
26,875
Opera..."

[2] Microsoft 10-K (FY2025) (similarity: 0.550)
    "f determining the geographic source of the revenue.
84
PART II
Item 8
Revenue, classified by significant product and service offerings, was as follows:
(In millions)
Year Ended June 30,
2025
2024
2023
Server products and cloud services
$
98,435
$
79,828
$
65,007
Microsoft 365 Commercial products and..."

[3] Microsoft 10-K (FY2025) (similarity: 0.538)
    ",820
13%
C

In [ ]:
# Known companies in our document set
KNOWN_COMPANIES = ['Apple', 'Microsoft', 'Tesla']
def detect_mentioned_company(query, known_companies):
    """Very simple check: does the query explicitly name a company we have?
    Returns the matched company name, or None if no known company is mentioned."""
    query_lower = query.lower()
    for company in known_companies:
        if company.lower() in query_lower:
            return company
    return None

def answer_question_filtered(query, index, chunks, metadata, model, k=3, similarity_threshold=0.35):
    results, confident = retrieve_with_citations(query, index, chunks, metadata, model, k=k, similarity_threshold=similarity_threshold)
    print(f"QUERY: {query}\n")
    if not confident:
        print("→ ANSWER: Not found in the provided documents.")
        print(f"  (Best match similarity: {results[0]['score']:.3f}, below threshold)")
        return


    mentioned_company = detect_mentioned_company(query, KNOWN_COMPANIES)
    top_result_company = results[0]['company']
    if mentioned_company and mentioned_company != top_result_company:
        print(f"→ ANSWER: Not found in the provided documents.")
        print(f"  (Query mentions '{mentioned_company}', but no matching content found for that company —")
        print(f"   closest match was from {top_result_company}'s filing, which would be a wrong-company answer.)")
        return

    print(" Retrieved supporting evidence:\n")
    for r in results:
        print(f"[{r['rank']}] {r['filing']} (similarity: {r['score']:.3f})")
        print(f"    \"{r['chunk'][:300]}...\"")
        print()

# Re-test the Amazon case that failed before
answer_question_filtered("What was Amazon's total net sales revenue?", combined_index, all_chunks, all_metadata, embed_model)
print("\n" + "="*60 + "\n")
# Confirm Apple still works correctly
answer_question_filtered("What was Apple's total net sales revenue?", combined_index, all_chunks, all_metadata, embed_model)

QUERY: What was Amazon's total net sales revenue?

 Retrieved supporting evidence:

[1] Microsoft 10-K (FY2025) (similarity: 0.567)
    "revenue, operating expenses, and operating income were as follows during the periods presented:
(In millions)
Year Ended June 30,
2025
2024
2023
Productivity and Business Processes
Revenue
$
120,810
$
106,820
$
94,151
Cost of revenue
22,422
19,611
17,202
Operating expenses
28,615
27,548
26,875
Opera..."

[2] Microsoft 10-K (FY2025) (similarity: 0.550)
    "f determining the geographic source of the revenue.
84
PART II
Item 8
Revenue, classified by significant product and service offerings, was as follows:
(In millions)
Year Ended June 30,
2025
2024
2023
Server products and cloud services
$
98,435
$
79,828
$
65,007
Microsoft 365 Commercial products and..."

[3] Microsoft 10-K (FY2025) (similarity: 0.538)
    ",820
13%
Cost of revenue
22,422
19,611
14%
Operating expenses
28,615
27,548
4%
Operating Income
$
69,773
$
59,661
17%
Intelligent Cloud
Revenue
$

In [ ]:
#correcting filter
KNOWN_COMPANIES = ['Apple', 'Microsoft', 'Tesla']
OTHER_COMPANIES = ['Amazon', 'Google', 'Alphabet', 'Meta', 'Netflix', 'Nvidia', 'Samsung', 'IBM', 'Intel']
def detect_mentioned_company(query, known_companies, other_companies):
    query_lower = query.lower()
    for company in known_companies:
        if company.lower() in query_lower:
            return company, True
    for company in other_companies:
        if company.lower() in query_lower:
            return company, False
    return None, None

def answer_question_filtered(query, index, chunks, metadata, model, k=3, similarity_threshold=0.35):
    results, confident = retrieve_with_citations(query, index, chunks, metadata, model, k=k, similarity_threshold=similarity_threshold)
    print(f"QUERY: {query}\n")
    if not confident:
        print("ANSWER: Not found in the provided documents.")
        print(f"  (Best match similarity: {results[0]['score']:.3f}, below threshold)")
        return

    mentioned_company, is_known = detect_mentioned_company(query, KNOWN_COMPANIES, OTHER_COMPANIES)

    #Case1 query names a company we simply don't have documents for
    if mentioned_company and not is_known:
        print(f"ANSWER: Not found in the provided documents.")
        print(f"  (Query asks about '{mentioned_company}', which is not in this document set — only Apple, Microsoft, and Tesla filings are available.)")
        return

    #Case2 query names a known company but top match is from a DIFFERENT known company
    top_result_company = results[0]['company']
    if mentioned_company and is_known and mentioned_company != top_result_company:
        print(f"ANSWER: Not found in the provided documents.")
        print(f"  (Query mentions '{mentioned_company}', but the closest match was from {top_result_company}'s filing.)")
        return

    print("Retrieved supporting evidence:\n")
    for r in results:
        print(f"[{r['rank']}] {r['filing']} (similarity: {r['score']:.3f})")
        print(f"    \"{r['chunk'][:300]}...\"")
        print()

# Retest
answer_question_filtered("What was Amazon's total net sales revenue?", combined_index, all_chunks, all_metadata, embed_model)
print("\n" + "="*60 + "\n")
answer_question_filtered("What was Apple's total net sales revenue?", combined_index, all_chunks, all_metadata, embed_model)

QUERY: What was Amazon's total net sales revenue?

ANSWER: Not found in the provided documents.
  (Query asks about 'Amazon', which is not in this document set — only Apple, Microsoft, and Tesla filings are available.)


QUERY: What was Apple's total net sales revenue?

Retrieved supporting evidence:

[1] Apple 10-K (FY2025) (similarity: 0.651)
    "9
13
%
85,200
Total net sales
$
416,161
6
%
$
391,035
2
%
$
383,285
(1)
Services net sales include amortization of the deferred value of services bundled in the sales price of certain products.
iPhone
iPhone net sales increased during 2025 compared to 2024 due to higher net sales of Pro models.
Mac
..."

[2] Apple 10-K (FY2025) (similarity: 0.642)
    "efore transferring it to the customer. Therefore, the Company accounts for all third-party application–related sales on a net basis by recognizing in Services net sales only the commission it retains.
Apple Inc. | 2025 Form 10-K | 35
The following table shows disaggregated net sales, as well 

## Metadata filtering: catching wrong company answers

Initial test revealed a real failure: asking about "Amazon's total net sales revenue" (a
company not in our document set) returned Microsoft's revenue figures with high confidence
(0.567 similarity, above threshold) because pure semantic similarity has no concept of
*which company* a chunk describes, only that the text is "about revenue."

**Fix:** added a lightweight company name check before returning any answer:
- If the query names a company not in our 3 document set decline explicitly.
- If the query names one of our 3 companies, but the top retrieved chunk is from a
  *different* one of our 3 companies → decline (this catches a subtler mismatch case).

**Honest limitation:** this is a simple keyword match against a hardcoded company list, not
true named-entity recognition. It won't catch every possible misnamed entity, and a production
system would want proper NER or FAISS native metadata filtering (restricting the search itself
to a specific company's chunks) rather than a post hoc keyword check. This is a good example
of the "I don't know is better than a confidently wrong number" principle — the fix needed here
wasn't a better retriever, it was recognizing what the retriever *couldn't* tell us on its own.

In [ ]:
while True:
    query = input("Ask a question about Apple, Microsoft, or Tesla's 10-K (or type 'quit' to stop): ")
    if query.lower() == 'quit':
        break
    print()
    answer_question_filtered(query, combined_index, all_chunks, all_metadata, embed_model)
    print("\n" + "="*60 + "\n")

Ask a question about Apple, Microsoft, or Tesla's 10-K (or type 'quit' to stop): What was Tesla's total revenue for 2024?

QUERY: What was Tesla's total revenue for 2024?

Retrieved supporting evidence:

[1] Tesla 10-K (FY2024) (similarity: 0.584)
    "its mission and grow professionally, earning Tesla among the Top 100 Employers of Choice in the 2024 American Opportunity Index. As of December 31, 2024, our employee headcount worldwide was 125,665.
Employees can participate in Tesla stock ownership programs (of which 92% have been given the opport..."

[2] Tesla 10-K (FY2024) (similarity: 0.583)
    "estricted cash, end of period
$
17,037
$
17,189
$
16,924
Supplemental Non-Cash Investing and Financing Activities
Acquisitions of property and equipment included in liabilities
$
1,410
$
2,272
$
2,148
Supplemental Disclosures
Cash paid during the period for interest
$
277
$
126
$
152
Cash paid durin..."

[3] Tesla 10-K (FY2024) (similarity: 0.564)
    "current legislation, qualifying Tesla

In [ ]:
# Search Tesla's chunks directly for the word "Total revenues" to find the real chunk
for i, c in enumerate(tesla_chunks):
    if 'Total revenues' in c or 'total revenues' in c:
        print(f"Chunk #{i}:")
        print(c[:400])
        print()

Chunk #230:
ce and charging infrastructure.
In 2024, we deployed 31.4 GWh of energy storage products. We are focused on ramping the production and increasing the market penetration of our energy storage products.
In 2024, we recognized total revenues of $97.69 billion, representing an increase of $917 million compared to the prior year. In 2024, our net income attributable to common stockholders was $7.09 bil

Chunk #263:
s)
2024
2023
2022
$
%
$
%
Automotive sales
$
72,480
$
78,509
$
67,210
$
(6,029)
(8)
%
$
11,299
17
%
Automotive regulatory credits
2,763
1,790
1,776
973
54
%
14
1
%
Automotive leasing
1,827
2,120
2,476
(293)
(14)
%
(356)
(14)
%
Total automotive revenues
77,070
82,419
71,462
(5,349)
(6)
%
10,957
15
%
Services and other
10,534
8,319
6,091
2,215
27
%
2,228
37
%
Total automotive & services and other se

Chunk #321:
ales
$
72,480
$
78,509
$
67,210
Automotive regulatory credits
2,763
1,790
1,776
Automotive leasing
1,827
2,120
2,476
Total automotive revenues
77,070
82,419
71,

In [ ]:
query = "What was Tesla's total revenue for 2024?"
query_embedding = embed_model.encode([query])
query_embedding_norm = normalize(np.array(query_embedding)).astype('float32')

# Find chunk 230's position in the combined index and its actual similarity score
target_chunk = tesla_chunks[230]
target_embedding = embed_model.encode([target_chunk])
target_embedding_norm = normalize(np.array(target_embedding)).astype('float32')

similarity_to_target = float(np.dot(query_embedding_norm, target_embedding_norm.T)[0][0])
print(f"Similarity between query and the CORRECT chunk (#230): {similarity_to_target:.4f}")
print()
print("Compare to what actually won the top-3:")
print("  Rank 1 (headcount chunk): 0.584")
print("  Rank 2 (cash flow chunk): 0.583")
print("  Rank 3 (regulatory credits chunk): 0.564")

Similarity between query and the CORRECT chunk (#230): 0.5306

Compare to what actually won the top-3:
  Rank 1 (headcount chunk): 0.584
  Rank 2 (cash flow chunk): 0.583
  Rank 3 (regulatory credits chunk): 0.564


In [ ]:
answer_question_filtered("What was Tesla's total revenue for 2024?", combined_index, all_chunks, all_metadata, embed_model, k=8)

QUERY: What was Tesla's total revenue for 2024?

Retrieved supporting evidence:

[1] Tesla 10-K (FY2024) (similarity: 0.584)
    "its mission and grow professionally, earning Tesla among the Top 100 Employers of Choice in the 2024 American Opportunity Index. As of December 31, 2024, our employee headcount worldwide was 125,665.
Employees can participate in Tesla stock ownership programs (of which 92% have been given the opport..."

[2] Tesla 10-K (FY2024) (similarity: 0.583)
    "estricted cash, end of period
$
17,037
$
17,189
$
16,924
Supplemental Non-Cash Investing and Financing Activities
Acquisitions of property and equipment included in liabilities
$
1,410
$
2,272
$
2,148
Supplemental Disclosures
Cash paid during the period for interest
$
277
$
126
$
152
Cash paid durin..."

[3] Tesla 10-K (FY2024) (similarity: 0.564)
    "current legislation, qualifying Tesla customers may receive up to $7,500 in federal tax credits for the purchase of qualified electric vehicles in the U.S. thro

## Retrieval limitation: "similar" isn't the same as "relevant"

Testing "What was Tesla's total revenue for 2024?" at k=3 returned three plausible looking
Tesla-related financial chunks (headcount, cash flow supplementals, regulatory credits) none
of which actually answered the question. The chunk containing the real answer ($97.69 billion
total revenue) scored 0.5306 similarity close to, but below, the three chunks that won
(0.584, 0.583, 0.564).

**Root cause:** the correct chunk discusses energy storage deployment before mentioning total
revenue, so its embedding reflects a mix of topics rather than being narrowly "about revenue"
diluting its similarity score relative to chunks that are more singularly financial-figure-shaped,
even though those chunks don't contain the specific answer.

**Retesting at k=8:** the correct chunk appeared at rank 7 of 8 confirming this was a
near-miss, not a fundamental retrieval failure. Increasing k catches it, but at the cost of
including several more irrelevant chunks in the context an LLM would have to sift through.

**Takeaway:** similarity score is not a proxy for "contains the answer." A chunk can rank
highly for being topically adjacent to a question without ever addressing it directly, and
the reverse can also be true. This is a genuine, inherent limitation of pure vector similarity
retrieval not something either chunking strategy tested this week fixes on its own. A more
robust system would likely combine this with keyword/exact-match search (hybrid retrieval) or
retrieve more candidates and rerank them with a more precise model before generating an answer.

In [ ]:
while True:
    query = input("Ask a question about Apple, Microsoft, or Tesla's 10-K (or type 'quit' to stop): ")
    if query.lower() == 'quit':
        break
    print()
    answer_question_filtered(query, combined_index, all_chunks, all_metadata, embed_model)
    print("\n" + "="*60 + "\n")

Ask a question about Apple, Microsoft, or Tesla's 10-K (or type 'quit' to stop): What was Microsoft's revenue from cloud services?

QUERY: What was Microsoft's revenue from cloud services?

Retrieved supporting evidence:

[1] Microsoft 10-K (FY2025) (similarity: 0.738)
    "f determining the geographic source of the revenue.
84
PART II
Item 8
Revenue, classified by significant product and service offerings, was as follows:
(In millions)
Year Ended June 30,
2025
2024
2023
Server products and cloud services
$
98,435
$
79,828
$
65,007
Microsoft 365 Commercial products and..."

[2] Microsoft 10-K (FY2025) (similarity: 0.736)
    "g, and selling our other products and services; and income taxes.
Highlights from fiscal year 2025 compared with fiscal year 2024 included:
•
Microsoft Cloud revenue increased 23% to $168.9 billion.
•
Microsoft 365 Commercial products and cloud services revenue increased 14% driven by Microsoft 365 ..."

[3] Microsoft 10-K (FY2025) (similarity: 0.714)
    "•
Opera

In [ ]:
# Confirm Apples filing actually has R&D spending data
for i, c in enumerate(apple_chunks_recursive):
    if 'research and development' in c.lower() and '$' in c:
        print(f"Chunk #{i}:")
        print(c[:400])
        print()

Chunk #161:
Products gross margin percentage decreased during 2025 compared to 2024 primarily due to a different mix of products and tariff costs, partially offset by other favorable costs.
Services Gross Margin
Services gross margin increased during 2025 compared to 2024 primarily due to higher Services net sales and a different mix of services.
Services gross margin percentage increased during 2025 compared

Chunk #162:
enses for 2025, 2024 and 2023 were as follows (dollars in millions):
2025
Change
2024
Change
2023
Research and development
$
34,550
10
%
$
31,370
5
%
$
29,915
Percentage of total net sales
8
%
8
%
8
%
Selling, general and administrative
$
27,601
6
%
$
26,097
5
%
$
24,932
Percentage of total net sales
7
%
7
%
7
%
Total operating expenses
$
62,151
8
%
$
57,467
5
%
$
54,847
Percentage of total net sa

Chunk #184:
3
Net sales:
Products
$
307,003
$
294,866
$
298,085
Services
109,158
96,169
85,200
Total net sales
416,161
391,035
383,285
Cost of sales:
Products
194,116
185,2

In [ ]:
# Each filing only covers these fiscal years used to catch out of range questions
FILING_YEAR_COVERAGE = {
    'Apple 10-K (FY2025)': [2023, 2024, 2025],
    'Microsoft 10-K (FY2025)': [2023, 2024, 2025],
    'Tesla 10-K (FY2024)': [2022, 2023, 2024],
}

def detect_year_in_query(query):
    match = re.search(r'\b(19|20)\d{2}\b', query)
    return int(match.group()) if match else None

def answer_question_v2(query, index, chunks, metadata, model, k=5, similarity_threshold=0.35):
    results, confident = retrieve_with_citations(query, index, chunks, metadata, model, k=k, similarity_threshold=similarity_threshold)
    print(f"QUERY: {query}\n")
    if not confident:
        print("→ ANSWER: Not found in the provided documents.")
        print(f"  (Best match similarity: {results[0]['score']:.3f}, below threshold)")
        return

    mentioned_company, is_known = detect_mentioned_company(query, KNOWN_COMPANIES, OTHER_COMPANIES)
    if mentioned_company and not is_known:
        print(f"→ ANSWER: Not found in the provided documents.")
        print(f"  (Query asks about '{mentioned_company}', which is not in this document set.)")
        return

    #check ALL top-k results for a company match, not just rank 1
    if mentioned_company and is_known:
        matching_results = [r for r in results if r['company'] == mentioned_company]
        if not matching_results:
            print(f"→ ANSWER: Not found in the provided documents.")
            print(f"  (Query mentions '{mentioned_company}', but none of the top {k} matches came from that filing.)")
            return
        results = matching_results  # only show results from the correct company

    #check for a year outside this filing's coverage
    query_year = detect_year_in_query(query)
    if query_year:
        top_filing = results[0]['filing']
        covered_years = FILING_YEAR_COVERAGE.get(top_filing, [])
        if query_year not in covered_years:
            print(f"→ ANSWER: Not found in the provided documents.")
            print(f"  (Query asks about {query_year}, but {top_filing} only covers {min(covered_years)}-{max(covered_years)}.)")
            return

    print("→ Retrieved supporting evidence:\n")
    for r in results:
        print(f"[{r['rank']}] {r['filing']} (similarity: {r['score']:.3f})")
        print(f"    \"{r['chunk'][:300]}...\"")
        print()

# Retest both problem cases
answer_question_v2("How much did Apple spend on research and development?", combined_index, all_chunks, all_metadata, embed_model)
print("\n" + "="*60 + "\n")
answer_question_v2("What was Apple's revenue in 2010?", combined_index, all_chunks, all_metadata, embed_model)

QUERY: How much did Apple spend on research and development?

→ Retrieved supporting evidence:

[2] Apple 10-K (FY2025) (similarity: 0.558)
    "seek to compete primarily through aggressive pricing and very low cost structures, and by imitating the Company’s products and infringing on its intellectual property.
Apple Inc. | 2025 Form 10-K | 2
The Company’s ability to compete successfully depends heavily on ensuring the continuing and timely ..."

[3] Apple 10-K (FY2025) (similarity: 0.554)
    "entiating its business and that its success does depend in part on such ownership, the Company relies primarily on the innovative skills, technical competence and marketing abilities of its personnel.
The Company regularly files patent, design, copyright and trademark applications to protect innovat..."

[4] Apple 10-K (FY2025) (similarity: 0.520)
    "s instead of components customized to meet the Company’s requirements, further limiting the Company’s ability to obtain sufficient quantities of 

In [ ]:
query = "How much did Apple spend on research and development?"
query_embedding = embed_model.encode([query])
query_embedding_norm = normalize(np.array(query_embedding)).astype('float32')
target_chunk = apple_chunks_recursive[162]
target_embedding = embed_model.encode([target_chunk])
target_embedding_norm = normalize(np.array(target_embedding)).astype('float32')
similarity_to_target = float(np.dot(query_embedding_norm, target_embedding_norm.T)[0][0])
print(f"Similarity between query and the CORRECT chunk (#162): {similarity_to_target:.4f}")
# Also check with a much higher k to see where it actually lands
results, _ = retrieve_with_citations(query, combined_index, all_chunks, all_metadata, embed_model, k=30, similarity_threshold=0.0)
for r in results:
    if r['company'] == 'Apple' and 'Research and development' in r['chunk']:
        print(f"Found at rank {r['rank']} of 30, similarity {r['score']:.4f}")
        break
else:
    print("Not found even in top 30 Apple/all results")

Similarity between query and the CORRECT chunk (#162): 0.5091
Found at rank 8 of 30, similarity 0.5091


## Recurring limitation: number dense chunks rank poorly against natural language questions

This pattern showed up twice independently:

| Question | Correct chunk's rank | Similarity score |
|---|---|---|
| Tesla's total revenue for 2024 | 7 of 8 | 0.5306 |
| Apple's R&D spending | 8 of 30 | 0.5091 |

Both correct chunks are financial-statement-style text — dense with numbers and labels, light
on descriptive sentences. A natural-language question like "how much did Apple spend on R&D"
embeds closer to chunks that *discuss* R&D or competition in prose, even when those chunks
don't contain the actual figure, than to the bare line "Research and development $34,550."

**This is a real, structural limitation of naive top k semantic retrieval on financial tables**,
not something either chunking strategy from earlier in this notebook fixes — the chunk
boundaries were fine in both cases (the number was intact and correctly attributed to its
label), the retrieval ranking itself is what missed it. A production system would likely need
one of:
- **Hybrid retrieval** — combine semantic search with keyword/exact-match search (e.g., BM25),
  so a literal match on "research and development" boosts the table chunk regardless of how it
  embeds.
- **A larger k with a reranking step** — retrieve a wide net (k=20-30), then use a more precise
  (and more expensive) model to rescore and reorder just those candidates.
- **Query rewriting** — reformulate "how much did X spend on Y" into something closer to the
  table's own phrasing before embedding it.


In [ ]:
#detecting the section headers and taging chunks with their sections
import re
#10K filings use a standard numbering scheme for major sections ("Items")
ITEM_PATTERN = re.compile(r'\bItem\s+(\d+[A-Z]?)\.?\s*[—\-–]?\s*([A-Z][a-zA-Z, ]{3,60})', re.IGNORECASE)

def tag_chunks_with_sections(chunks, full_text):
    #For each chunk, finds the nearest preceding 'Item X Section Name' header in the original document, so every chunk knows what section it belongs to

    #Firstly finding all section header positions in the full text
    section_markers = []
    for match in ITEM_PATTERN.finditer(full_text):
        section_markers.append((match.start(), match.group(0).strip()))

    tagged = []
    search_pos = 0
    for chunk in chunks:
        chunk_pos = full_text.find(chunk[:50], search_pos)  # approximate location of this chunk
        if chunk_pos == -1:
            chunk_pos = search_pos  #fallback if exact match fails

        # Find the last section header that appears BEFORE this chunk's position
        current_section = "Unknown section"
        for pos, header in section_markers:
            if pos <= chunk_pos:
                current_section = header
            else:
                break

        tagged.append(current_section)
        search_pos = chunk_pos

    return tagged

# Test on Apple first
apple_sections = tag_chunks_with_sections(apple_chunks_recursive, apple_text)
print(f"Tagged {len(apple_sections)} Apple chunks with sections")
print()
print("Sample of unique sections found:")
for s in sorted(set(apple_sections))[:15]:
    print(" -", s)

Tagged 300 Apple chunks with sections

Sample of unique sections found:
 - Item 1.
Business
 - Item 1.    Business
 - Item 11.
Executive Compensation
 - Item 12.    Security Ownership of Certain Beneficial Owners and Managemen
 - Item 15.    Exhibit and Financial Statement Schedules
 - Item 16.    Form
 - Item 1A of this Form
 - Item 1A.    Risk Factors
 - Item 1C.    Cybersecurity
 - Item 2.    Properties
 - Item 3.    Legal Proceedings
 - Item 5.    Market for Registrant
 - Item 601 of Regulation S
 - Item 7 of the Company
 - Item 7 of this Form


In [ ]:
#a bit more better detection
# Only match "Item X." at the START of a line which are real headers not mid sentence references
ITEM_PATTERN = re.compile(r'^Item\s+(\d+[A-Z]?)\.\s*([A-Z][a-zA-Z, ]{3,60})', re.IGNORECASE | re.MULTILINE)
def tag_chunks_with_sections(chunks, full_text):
    section_markers = []
    for match in ITEM_PATTERN.finditer(full_text):
        # Normalize whitespace so "Item 1.    Business" and "Item 1. Business" match as the same section
        clean_header = re.sub(r'\s+', ' ', match.group(0).strip())
        section_markers.append((match.start(), clean_header))

    tagged = []
    search_pos = 0
    for chunk in chunks:
        chunk_pos = full_text.find(chunk[:50], search_pos)
        if chunk_pos == -1:
            chunk_pos = search_pos

        current_section = "Unknown section"
        for pos, header in section_markers:
            if pos <= chunk_pos:
                current_section = header
            else:
                break

        tagged.append(current_section)
        search_pos = chunk_pos

    return tagged

apple_sections = tag_chunks_with_sections(apple_chunks_recursive, apple_text)
print(f"Tagged {len(apple_sections)} Apple chunks with sections")
print()
print("Unique sections found:")
for s in sorted(set(apple_sections)):
    print(" -", s)

Tagged 300 Apple chunks with sections

Unique sections found:
 - Item 1. Business
 - Item 11. Executive Compensation
 - Item 12. Security Ownership of Certain Beneficial Owners and Managemen
 - Item 15. Exhibit and Financial Statement Schedules
 - Item 16. Form
 - Item 1A. Risk Factors
 - Item 1C. Cybersecurity
 - Item 2. Properties
 - Item 3. Legal Proceedings
 - Item 5. Market for Registrant
 - Item 7. Management
 - Item 7A. Quantitative and Qualitative Disclosures About Market Risk
 - Item 8. Financial Statements and Supplementary Data
 - Item 9A. Controls and Procedures
 - Item 9B. Other Information
 - Item 9C. Disclosure Regarding Foreign Jurisdictions that Prevent Inspe
 - Unknown section


In [ ]:
microsoft_sections = tag_chunks_with_sections(microsoft_chunks, microsoft_text)
tesla_sections = tag_chunks_with_sections(tesla_chunks, tesla_text)

# Rebuild all_metadata with section info included
all_metadata = []

for section in apple_sections:
    all_metadata.append({'company': 'Apple', 'filing': 'Apple 10-K (FY2025)', 'section': section})

for section in microsoft_sections:
    all_metadata.append({'company': 'Microsoft', 'filing': 'Microsoft 10-K (FY2025)', 'section': section})

for section in tesla_sections:
    all_metadata.append({'company': 'Tesla', 'filing': 'Tesla 10-K (FY2024)', 'section': section})

print(f"Total metadata entries: {len(all_metadata)}")
print("Sample:", all_metadata[162])  # should show Apple's R&D chunk's section

Total metadata entries: 1345
Sample: {'company': 'Apple', 'filing': 'Apple 10-K (FY2025)', 'section': 'Item 7. Management'}


In [ ]:
#updating citations functions to include sections
def answer_question_v3(query, index, chunks, metadata, model, k=8, similarity_threshold=0.35):
    results, confident = retrieve_with_citations(query, index, chunks, metadata, model, k=k, similarity_threshold=similarity_threshold)
    print(f"QUERY: {query}\n")
    if not confident:
        print("→ ANSWER: Not found in the provided documents.")
        print(f"  (Best match similarity: {results[0]['score']:.3f}, below threshold)")
        return

    mentioned_company, is_known = detect_mentioned_company(query, KNOWN_COMPANIES, OTHER_COMPANIES)
    if mentioned_company and not is_known:
        print(f"→ ANSWER: Not found in the provided documents.")
        print(f"  (Query asks about '{mentioned_company}', which is not in this document set.)")
        return

    if mentioned_company and is_known:
        matching_results = [r for r in results if r['company'] == mentioned_company]
        if not matching_results:
            print(f"→ ANSWER: Not found in the provided documents.")
            print(f"  (Query mentions '{mentioned_company}', but none of the top {k} matches came from that filing.)")
            return
        results = matching_results

    query_year = detect_year_in_query(query)
    if query_year:
        top_filing = results[0]['filing']
        covered_years = FILING_YEAR_COVERAGE.get(top_filing, [])
        if query_year not in covered_years:
            print(f"→ ANSWER: Not found in the provided documents.")
            print(f"  (Query asks about {query_year}, but {top_filing} only covers {min(covered_years)}-{max(covered_years)}.)")
            return

    print("→ Retrieved supporting evidence:\n")
    for r in results:
        print(f"[{r['rank']}] {r['filing']} — {r['section']} (similarity: {r['score']:.3f})")
        print(f"    \"{r['chunk'][:300]}...\"")
        print()

# Need to update retrieve_with_citations to also carry 'section' through
def retrieve_with_citations(query, index, chunks, metadata, model, k=3, similarity_threshold=0.35):
    query_embedding = model.encode([query])
    query_embedding_norm = normalize(np.array(query_embedding)).astype('float32')
    scores, indices = index.search(query_embedding_norm, k)

    results = []
    for rank, idx in enumerate(indices[0]):
        score = scores[0][rank]
        results.append({
            'rank': rank + 1,
            'score': float(score),
            'chunk': chunks[idx],
            'company': metadata[idx]['company'],
            'filing': metadata[idx]['filing'],
            'section': metadata[idx]['section'],
        })

    best_score = results[0]['score'] if results else 0
    confident = best_score >= similarity_threshold

    return results, confident

# Test with a real question
answer_question_v3("How much did Apple spend on research and development?", combined_index, all_chunks, all_metadata, embed_model)

QUERY: How much did Apple spend on research and development?

→ Retrieved supporting evidence:

[2] Apple 10-K (FY2025) — Item 1. Business (similarity: 0.558)
    "seek to compete primarily through aggressive pricing and very low cost structures, and by imitating the Company’s products and infringing on its intellectual property.
Apple Inc. | 2025 Form 10-K | 2
The Company’s ability to compete successfully depends heavily on ensuring the continuing and timely ..."

[3] Apple 10-K (FY2025) — Item 1. Business (similarity: 0.554)
    "entiating its business and that its success does depend in part on such ownership, the Company relies primarily on the innovative skills, technical competence and marketing abilities of its personnel.
The Company regularly files patent, design, copyright and trademark applications to protect innovat..."

[4] Apple 10-K (FY2025) — Item 8. Financial Statements and Supplementary Data (similarity: 0.520)
    "s instead of components customized to meet the Compan

In [ ]:
#Auto-detect company name from each document
def detect_company_name(text, filename):
    #Extract the company name from a 10-K's cover page. 10-Ks reliably state '[Company Name] Inc.' or similar near the top, right after the Commission File Number line."""
    #Look in just the first 3000 characters (cover page)
    cover_page = text[:3000]
    #Common patterns: "Apple Inc.", "Microsoft Corporation", "Tesla, Inc."
    match = re.search(r'\n([A-Z][a-zA-Z0-9&.,\' ]{2,40}?(?:Inc\.|Corporation|Corp\.|LLC|Company|Co\.))\s*\n', cover_page)
    if match:
        return match.group(1).strip()

    #Fallback: use the filename if pattern-matching fails
    return filename.replace('_10k.htm', '').replace('_', ' ').title()

# Test on all 3 documents
for name, text, filename in [('Apple', apple_text, 'apple_10k.htm'),
                                ('Microsoft', microsoft_text, 'microsoft_10k.htm'),
                                ('Tesla', tesla_text, 'tesla_10k.htm')]:
    detected = detect_company_name(text, filename)
    print(f"{filename}: detected company name = '{detected}'")

apple_10k.htm: detected company name = 'Apple Inc.'
microsoft_10k.htm: detected company name = 'Smaller Reporting Company'
tesla_10k.htm: detected company name = 'Tesla, Inc.'


In [ ]:
#More reliable company name detection using a fixed anchor phrase
def detect_company_name(text, filename):
#10-Ks reliably include the phrase '(Exact name of Registrant as specified in its charter)' immediately after the company name on the cover page. Anchoring to this phrase is far more reliable than a loose name pattern

    cover_page = text[:3000]
    anchor = re.search(r'\(Exact name of [Rr]egistrant', cover_page)
    if anchor:
        # Look at the text just BEFORE the anchor that's where the company name sits
        preceding_text = cover_page[:anchor.start()]
        lines = [l.strip() for l in preceding_text.split('\n') if l.strip()]
        if lines:
            return lines[-1]  # the line immediately before the anchor phrase

    # Fallback: use filename if the anchor phrase isn't found
    return filename.replace('_10k.htm', '').replace('_', ' ').title()

for name, text, filename in [('Apple', apple_text, 'apple_10k.htm'),
                                ('Microsoft', microsoft_text, 'microsoft_10k.htm'),
                                ('Tesla', tesla_text, 'tesla_10k.htm')]:
    detected = detect_company_name(text, filename)
    print(f"{filename}: detected company name = '{detected}'")

apple_10k.htm: detected company name = 'Apple Inc.'
microsoft_10k.htm: detected company name = 'Microsoft'
tesla_10k.htm: detected company name = 'Tesla, Inc.'


In [ ]:
# Generalized ingestion pipeline (works for any folder of 10Ks)
import glob

def ingest_document_folder(folder_path):
#Process every .htm file in a folder: extract text, detect company name, chunk it, tag sections no hardcoded company names anywhere

    all_chunks_new=[]
    all_metadata_new=[]
    company_registry=[] #tracks which companies we actually found, dynamically
    filepaths = glob.glob(f'{folder_path}/*.htm')
    print(f"Found {len(filepaths)} documents in {folder_path}")
    for filepath in filepaths:
        filename = filepath.split('/')[-1]
        text = extract_clean_text(filepath)
        company_name = detect_company_name(text, filename)
        company_registry.append(company_name)

        chunks = recursive_chunks(text, chunk_size=1000, overlap=200)
        sections = tag_chunks_with_sections(chunks, text)

        for chunk, section in zip(chunks, sections):
            all_chunks_new.append(chunk)
            all_metadata_new.append({
                'company': company_name,
                'filing': f'{company_name} ({filename})',
                'section': section
            })

        print(f"  {filename}: detected as '{company_name}', {len(chunks)} chunks")

    return all_chunks_new, all_metadata_new, company_registry

#rebuild everything using the generalized pipeline instead of hardcoded variables
all_chunks, all_metadata, KNOWN_COMPANIES = ingest_document_folder('filings')
print()
print(f"Total chunks: {len(all_chunks)}")
print(f"Dynamically detected companies: {KNOWN_COMPANIES}")

Found 3 documents in filings
  microsoft_10k.htm: detected as 'Microsoft', 471 chunks
  apple_10k.htm: detected as 'Apple Inc.', 300 chunks
  tesla_10k.htm: detected as 'Tesla, Inc.', 574 chunks

Total chunks: 1345
Dynamically detected companies: ['Microsoft', 'Apple Inc.', 'Tesla, Inc.']


In [ ]:
#Rebuilding the index and retest
all_embeddings = embed_model.encode(all_chunks, show_progress_bar=True)
all_embeddings_norm = normalize(np.array(all_embeddings)).astype('float32')
combined_index = faiss.IndexFlatIP(dimension)
combined_index.add(all_embeddings_norm)
print("Rebuilt FAISS index with", combined_index.ntotal, "vectors")
#Update OTHER_COMPANIES and FILING_YEAR_COVERAGE to use the new dynamic filing names
OTHER_COMPANIES=['Amazon', 'Google', 'Alphabet', 'Meta', 'Netflix', 'Nvidia', 'Samsung', 'IBM', 'Intel']
FILING_YEAR_COVERAGE={
    metadata['filing']: [2023, 2024, 2025] if 'apple' in metadata['filing'].lower() or 'microsoft' in metadata['filing'].lower()
                          else [2022, 2023, 2024]
    for metadata in all_metadata
}

#rerun our key test cases
answer_question_v3("How much did Apple spend on research and development?", combined_index, all_chunks, all_metadata, embed_model)
print("\n" + "="*60 + "\n")
answer_question_v3("What was Amazon's total net sales revenue?", combined_index, all_chunks, all_metadata, embed_model)

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Rebuilt FAISS index with 1345 vectors
QUERY: How much did Apple spend on research and development?

→ Retrieved supporting evidence:

[1] Microsoft (microsoft_10k.htm) — ITEM 7. MANAGEMENT (similarity: 0.602)
    "quisition.
OPERATING EXPENSES
Research and Development
(In millions, except percentages)
2025
2024
Percentage
Change
Research and development
$
32,488
$
29,510
10%
As a percent of revenue
12%
12%
0ppt
Research and development expenses include payroll, employee benefits, stock-based compensation expe..."

[2] Apple Inc. (apple_10k.htm) — Item 1. Business (similarity: 0.558)
    "seek to compete primarily through aggressive pricing and very low cost structures, and by imitating the Company’s products and infringing on its intellectual property.
Apple Inc. | 2025 Form 10-K | 2
The Company’s ability to compete successfully depends heavily on ensuring the continuing and timely ..."

[3] Apple Inc. (apple_10k.htm) — Item 1. Business (similarity: 0.554)
    "entiating its business a

In [ ]:
LEGAL_SUFFIXES = [' Inc.', ', Inc.', ' Corporation', ' Corp.', ' LLC', ', LLC', ' Company', ' Co.']

def normalize_company_name(name):
    """Strip legal suffixes for matching purposes only — display names stay untouched."""
    clean = name
    for suffix in LEGAL_SUFFIXES:
        clean = clean.replace(suffix, '')
    return clean.strip()

def detect_mentioned_company(query, known_companies, other_companies):
    query_lower = query.lower()
    for company in known_companies:
        normalized = normalize_company_name(company)
        if normalized.lower() in query_lower:
            return company, True  # return the FULL name for filtering but matched on the normalized version
    for company in other_companies:
        if company.lower() in query_lower:
            return company, False
    return None, None

# Re-test the case that just broke
answer_question_v3("How much did Apple spend on research and development?", combined_index, all_chunks, all_metadata, embed_model)

QUERY: How much did Apple spend on research and development?

→ Retrieved supporting evidence:

[2] Apple Inc. (apple_10k.htm) — Item 1. Business (similarity: 0.558)
    "seek to compete primarily through aggressive pricing and very low cost structures, and by imitating the Company’s products and infringing on its intellectual property.
Apple Inc. | 2025 Form 10-K | 2
The Company’s ability to compete successfully depends heavily on ensuring the continuing and timely ..."

[3] Apple Inc. (apple_10k.htm) — Item 1. Business (similarity: 0.554)
    "entiating its business and that its success does depend in part on such ownership, the Company relies primarily on the innovative skills, technical competence and marketing abilities of its personnel.
The Company regularly files patent, design, copyright and trademark applications to protect innovat..."

[4] Apple Inc. (apple_10k.htm) — Item 8. Financial Statements and Supplementary Data (similarity: 0.520)
    "s instead of components customiz

In [ ]:
#live Questions and answers using generalized pipline
while True:
    query = input("Ask a question about the filings in this folder (or type 'quit' to stop): ")
    if query.lower() == 'quit':
        break
    print()
    answer_question_v3(query, combined_index, all_chunks, all_metadata, embed_model, k=8)
    print("\n" + "="*60 + "\n")

Ask a question about the filings in this folder (or type 'quit' to stop): What was Tesla's total revenue for 2024?

QUERY: What was Tesla's total revenue for 2024?

→ Retrieved supporting evidence:

[1] Tesla, Inc. (tesla_10k.htm) — ITEM 1. BUSINESS (similarity: 0.584)
    "its mission and grow professionally, earning Tesla among the Top 100 Employers of Choice in the 2024 American Opportunity Index. As of December 31, 2024, our employee headcount worldwide was 125,665.
Employees can participate in Tesla stock ownership programs (of which 92% have been given the opport..."

[2] Tesla, Inc. (tesla_10k.htm) — ITEM 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA (similarity: 0.583)
    "estricted cash, end of period
$
17,037
$
17,189
$
16,924
Supplemental Non-Cash Investing and Financing Activities
Acquisitions of property and equipment included in liabilities
$
1,410
$
2,272
$
2,148
Supplemental Disclosures
Cash paid during the period for interest
$
277
$
126
$
152
Cash paid durin..."

[3

In [ ]:
import pickle

# Save the FAISS index itself
faiss.write_index(combined_index, 'financial_qa_index.faiss')

# Save the chunks and metadata together (the index alone is useless without these)
with open('financial_qa_data.pkl', 'wb') as f:
    pickle.dump({
        'chunks': all_chunks,
        'metadata': all_metadata,
        'known_companies': KNOWN_COMPANIES,
        'filing_year_coverage': FILING_YEAR_COVERAGE,
    }, f)

print("Saved:")
print("  - financial_qa_index.faiss (the vector index)")
print("  - financial_qa_data.pkl (chunks + metadata + company registry)")

from google.colab import files
files.download('financial_qa_index.faiss')
files.download('financial_qa_data.pkl')

Saved:
  - financial_qa_index.faiss (the vector index)
  - financial_qa_data.pkl (chunks + metadata + company registry)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

loaded_index = faiss.read_index('financial_qa_index.faiss')
with open('financial_qa_data.pkl', 'rb') as f:
    loaded_data = pickle.load(f)

loaded_chunks = loaded_data['chunks']
loaded_metadata = loaded_data['metadata']
KNOWN_COMPANIES = loaded_data['known_companies']
FILING_YEAR_COVERAGE = loaded_data['filing_year_coverage']

print(f"Loaded index with {loaded_index.ntotal} vectors")
print(f"Loaded {len(loaded_chunks)} chunks, {len(loaded_metadata)} metadata entries")
print(f"Known companies: {KNOWN_COMPANIES}")
answer_question_v3("What was Tesla's total revenue for 2024?", loaded_index, loaded_chunks, loaded_metadata, embed_model)

Loaded index with 1345 vectors
Loaded 1345 chunks, 1345 metadata entries
Known companies: ['Microsoft', 'Apple Inc.', 'Tesla, Inc.']
QUERY: What was Tesla's total revenue for 2024?

→ Retrieved supporting evidence:

[1] Tesla, Inc. (tesla_10k.htm) — ITEM 1. BUSINESS (similarity: 0.584)
    "its mission and grow professionally, earning Tesla among the Top 100 Employers of Choice in the 2024 American Opportunity Index. As of December 31, 2024, our employee headcount worldwide was 125,665.
Employees can participate in Tesla stock ownership programs (of which 92% have been given the opport..."

[2] Tesla, Inc. (tesla_10k.htm) — ITEM 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA (similarity: 0.583)
    "estricted cash, end of period
$
17,037
$
17,189
$
16,924
Supplemental Non-Cash Investing and Financing Activities
Acquisitions of property and equipment included in liabilities
$
1,410
$
2,272
$
2,148
Supplemental Disclosures
Cash paid during the period for interest
$
277
$
126
$
152
Cash p

## Chunking Strategy Comparison:Fixed Size vs Recursive/Boundary Aware

Two chunking strategies were implemented and tested on real SEC 10 K filings (Apple,Microsoft,
Tesla 180K words combined):

**Strategy 1: Fixed size:** slices text into 1000 character windows with 200-character overlap,
with no regard for sentence, paragraph, or table boundaries.

**Strategy 2: Recursive/boundar aware:** groups text by paragraph up to the target chunk size,
falling back to sentence boundary splitting only when a paragraph is too long. Produced 300
chunks for Apple's filing vs 259 for fixed size slightly more chunks, since respecting
boundaries means less tight packing.

**Direct comparison on the same table** (Apple's segment revenue breakdown, $416,161M total net
sales): fixed-size chunking started mid word ("ng Performance..."cut from "Segment Operating
Performance"), and only preserved the table's header by chance, since the character count
boundary happened to land nearby. Recursive chunking started at a clean sentence boundary and
flowed naturally into the table with its header intact, by design rather than luck. This was
confirmed by a separate example (Apple's iPhone/Mac/iPad revenue table), where fixed-size
chunking *did* lose its header entirely proving the difference isn't a one off coincidence.

**Honest limitation:** neither strategy fully solves table structure. Both flatten actual tables
into a single run of numbers rather than preserving rows/columns recursive chunking's real
advantage is avoiding mid sentence/mid word cuts not fixing tables outright. A complete fix
would need table aware parsing (extracting `<table>` elements separately before chunking
surrounding prose).

## Retrieval quality: a recurring, documented limitation

Two independent test cases revealed the same failure pattern:

| Question | Correct chunk's rank (of k) | Similarity score |
|---|---|---|
| Tesla's total revenue for 2024 | 7 of 8 | 0.5306 |
| Apple's R&D spending | 8 of 30 | 0.5091 |

In both cases, the correct answer lived in a terse, number dense chunk (a financial table with
minimal surrounding prose), while several plausible but wrong chunks topically related but not
actually containing the answer outranked it. Retesting with a wider k (8, then 30) confirmed
the correct chunk was recoverable, just not within a typical top 3 cutoff.

**Root cause:** a chunk's embedding reflects everything in it. A table heavy chunk with little
descriptive language around the number embeds less precisely toward a natural language question
than a chunk that discusses the topic in prose, even if that prose chunk never states the actual
figure. **Similarity score is not the same as "contains the answer."**

**Distance metric comparison (L2vscosine):** tested identically on the same query set; both
metrics returned the same top 3 chunks in the same order for this embedding model
(`all-MiniLM-L6-v2`), since it produces vectors of fairly consistent magnitude. This isn't
guaranteed for every model L2 and cosine can diverge when magnitudes vary more  but for this
setup, metric choice didn't change retrieval outcomes.

## Metadata filtering: catching wrong company and wrong year answers

Two real failures were found and fixed during testing:

1. **Wrong company, high confidence.** Asking about "Amazon's total net sales revenue" (a
   company not in the document set) returned Microsoft's revenue figures at 0.567 similarity —
   above the confidence threshold, and confidently wrong. Fixed by checking whether the query
   names a company outside the known set, and separately, whether the top-k results actually
   include a match for the company named.
2. **Wrong year, no warning.** Asking "What was Apple's revenue in 2010?" returned 2025 revenue
   data with no indication the filing doesn't cover 2010 at all. Fixed with a lightweight
   year detection check against each filing's actual coverage range.

**A live bug found during generalization:** after refactoring to auto detect company names from
each document (rather than hardcoding Apple, Microsoft,Tesla),the wrong company filter
silently broke Apple Inc. (the autdetected name) no longer matched a query saying just
Apple, since the substring check required an exact match including the legal suffix. Fixed by
normalizing company names (stripping "Inc.", "Corporation," etc.) before matching, while keeping
full names for citation display. This is a good example of how generalizing a system can quietly
reintroduce bugs a more hardcoded version didn't have.

## Bottom line

The system correctly answers direct factual questions with accurate filing+section citations,
and correctly declines when a question is unanswerable (wrong company, wrong year, or entirely
unrelated). Its main known weakness is retrieval ranking on terse, table heavy content
addressable with hybrid (keyword+semantic) retrieval or a reranking step, neither of which
was implemented this week but is clearly the next improvement worth making.

## What could go wrong model risk note

**Retrieval quality**

- *Similarity not equal relevance.* Demonstrated twice with real evidence (Tesla revenue, Apple R&D):
  a chunk can rank below several topically related but wrong chunks even when it directly
  answers the question because its embedding reflects everything in the chunk not just the
  specific fact asked about. Increasing k catches these near misses but adds noise for a
  downstream LLM (or human) to sift through there's no free fix, only a trade-off to tune.
- *No hybrid retrieval.* Pure semantic search has no concept of exact keyword matches. A
  production system would benefit from combining this with keyword/BM25 style search, so a
  literal match on a term like research and development boosts the right chunk regardless of
  how its surrounding context happens to embed.
- *Tables are still flattened.* Neither chunking strategy preserves actual table structure
  (rows/columns) both produce a single run of numbers. The correct figure is usually present
  and correctly labeled, but a more complex multi column table could scramble which number
  belongs to which year/category without a human noticing.

**Metadata filtering**

- *Company matching is a simple keyword check, not true NER.* The filter only catches
  mismatches against a small hardcoded list of "other well known companies" (Amazon, Google,
  etc.). A question about a genuinely obscure company not on that list, and not in the document
  folder, would not be caught by this specific guard though the underlying similarity
  threshold would likely still catch it if the semantic match were weak enough.
- *Year detection is regex based and brittle.* It only catches an explicit 4-digit year in the
  query. A question like "What was Apple's revenue last year?" or "five years ago?" would
  not trigger the year range check at all, since there's no literal year to detect.
- *Generalizing introduced a real bug once already.* Auto detecting company names from
  documents (to satisfy "works on a new folder without touching code") broke the existing
  wrong company filter because Apple Inc no longer matched a query saying "Apple." This is
  worth remembering: every generalization is a chance to silently break an assumption made
  during the specific, hardcoded version.

**Section tagging**

- *Regex based header detection, not a structural parser.* Section boundaries are found via a
  pattern match on "Item X." at the start of a line. This works for standard 10K formatting,
  but a filing with unusual formatting, OCR artifacts (for scanned documents), or a different
  numbering convention could produce "Unknown section" or mismatched boundaries.

**General**

- *Static snapshot.* All three filings are a single point in time annual report. Nothing here
  handles updates if a company files an amended 10 K/A, or a new fiscal year's filing arrives,
  the index would need to be rebuilt from scratch; there's no incremental update mechanism.
- *No source document versioning.* If SEC's filing content is ever corrected or restated, this
  system has no way of knowing its saved index reflects outdated numbers.